# Project 01 (basic): SQL — fetching data where they live

**Goal:** practise the basic SQL building blocks (SELECT, WHERE, GROUP BY, JOIN,
HAVING) on a real database — and see along the way how SQL and pandas work together.

**Preparation** (once, in the folder `01-basic`, venv active):

```
python generate_db.py
```

This creates `datasets/shop.db`: a SQLite database with a fictional online shop —
**60 customers, 25 products, 800 orders** in three linked tables:

```
customers(customer_id, name, city, registered_on)
products(product_id, name, category, price)
orders(order_id, customer_id -> customers, product_id -> products, quantity, ordered_on)
```

Note the principle: an order stores only *references* (foreign keys) to the customer
and the product — names and prices appear exactly once, in their own tables
(normalisation). To answer "revenue per city" you have to **join** the tables —
which is exactly what you practise here.

**Reference to the script:** section 2.4 (SQL). A tip beforehand: 15 minutes of
https://sqlbolt.com.

## 1. Opening a connection

SQLite is built into Python; `pd.read_sql` runs a query and delivers the result
directly as a DataFrame. The helper function `q(...)` saves typing.

In [ ]:
import os
import sqlite3
import pandas as pd

PATH = "datasets/shop.db" if os.path.exists("datasets/shop.db") else "../datasets/shop.db"
con = sqlite3.connect(PATH)

def q(sql):
    """Runs an SQL query and returns a DataFrame."""
    return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type = 'table'")

## 2. SELECT, WHERE, ORDER BY, LIMIT

The first pattern: select rows, filter, sort, limit.

```sql
SELECT col1, col2 FROM table WHERE condition ORDER BY col DESC LIMIT 5
```

**Tasks:**
1. All products of the category 'Books' — name and price, by price descending.
2. How many products cost more than 50 EUR? (`COUNT(*)`)
3. The 5 most recent orders (by `ordered_on`).

In [ ]:
# Task 1:
# TODO: SELECT name, price FROM products WHERE ... ORDER BY ...

In [ ]:
# Tasks 2 and 3:
# TODO: expensive = q("SELECT COUNT(*) AS n FROM products WHERE ...")
# TODO: newest = q("SELECT * FROM orders ORDER BY ... DESC LIMIT 5")
print(newest)

# Mini check:
print(int(expensive["n"][0]) == 6)

## 3. GROUP BY — aggregating like groupby

**Tasks:**
1. How many products are there per category? (`COUNT(*)`, `GROUP BY`)
2. Average price per category, sorted descending (`AVG`, `ROUND(x, 2)`).
3. Orders per month: `strftime('%m', ordered_on)` extracts the month —
   is there a December peak?

In [ ]:
# TODO: products per category
# TODO: average price per category, descending
# TODO: months = orders per month, then plot as a bar chart

## 4. JOIN — linking tables

The very reason the data sit in three tables. The pattern:

```sql
SELECT ... FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN products  p ON p.product_id  = o.product_id
```

**Tasks:**
1. Number of orders per **city** (needs: orders joined with customers).
2. **Revenue** (= `p.price * o.quantity`) per **category**, descending.
3. The **top 3 customers** by revenue (name + revenue; needs all three tables!).

In [ ]:
# TODO: orders per city (orders JOIN customers)
# TODO: revenue_by_category = revenue per category, descending
# TODO: top3 = top 3 customers by revenue (all three tables)

# Mini checks:
print(revenue_by_category["category"][0] == "Electronics")   # category with the highest revenue
print(top3["name"][0] == "Emma Schulz")                      # top customer

## 5. LEFT JOIN — who is missing?

A (INNER) `JOIN` shows only customers who have also ordered. But marketing asks:
**"which customers have NEVER ordered?"** For that: `LEFT JOIN` (keep all customers,
the order columns are then `NULL`) plus `WHERE o.order_id IS NULL`.

**Task:** find these customers (name, city). There should be **5** of them.

In [ ]:
# TODO: dormant_customers = customers LEFT JOIN orders, keep those without an order
print(dormant_customers)
print(len(dormant_customers) == 5)

## 6. WHERE vs. HAVING

`WHERE` filters rows **before** the grouping, `HAVING` filters groups **afterwards** —
only there may `COUNT(*)` and friends appear (script 2.4, self-test question 7!).

**Task:** which cities have 150 or more orders? (Expected: 4 cities.)
First deliberately try `WHERE COUNT(*) >= 150` — read the error message.

In [ ]:
# TODO: first deliberately with WHERE COUNT(*) >= 150 and read the error
# TODO: cities = cities with at least 150 orders (HAVING!)
print(cities)
print(len(cities) == 4)

## 7. Division of labour: SQL + pandas

The rule of thumb from the script: **filter/join in SQL, analyse/plot in pandas.**
This is what the typical workflow looks like — SQL delivers the compact analysis
table, pandas does the rest (given as is):

In [ ]:
analysis = q("""SELECT o.ordered_on, c.city, p.category,
                       p.price * o.quantity AS revenue
                FROM orders o
                JOIN customers c ON c.customer_id = o.customer_id
                JOIN products  p ON p.product_id  = o.product_id""")
analysis["ordered_on"] = pd.to_datetime(analysis["ordered_on"])

pivot = analysis.pivot_table(index="city", columns="category",
                             values="revenue", aggfunc="sum").round(0)
print(pivot)
pivot.plot.bar(stacked=True, figsize=(8, 4), title="Revenue per city and category");

In [ ]:
con.close()   # always close the connection at the end

## Done — what you can do now

- SELECT / WHERE / ORDER BY / LIMIT, aggregation with GROUP BY
- join tables via foreign keys (INNER and LEFT) — and know when to use which
- tell WHERE and HAVING apart confidently
- carry SQL results seamlessly on into pandas

**Bonus tasks** (optional):
1. Revenue per month as an SQL query (strftime + JOIN + SUM) — and as a line plot.
2. Which customer bought the most *different* products? (`COUNT(DISTINCT ...)`)
3. Rewrite the city/orders query from section 4 entirely in pandas (merge + groupby)
   and compare the results.